# SurvFace 입력 변환 개발 대조

같은 Parent와 Carbon s7e의 10 epoch 일정을 사용해 stretch와 종횡비 유지 패딩의 효과를 측정합니다. 00–04 실행 없이 이 노트북에서 시작합니다.

In [ ]:
from pathlib import Path
import os, sys, json
repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "fine-tune/modules").is_dir())
sys.path.insert(0, str(repo / "fine-tune"))
from modules.survface_input_pilot import ARMS, SIZES, GATE, run_pilot, read_summary

DATASET_ROOT = Path(os.environ.get("FACE_DATASET_ROOT", "F:/face_datasets"))
MODEL_ROOT = Path(os.environ.get("FACE_MODEL_ROOT", "F:/models"))
RUN_ID = "paired_20261124_v1"  # 조회할 정확한 ID. 새 실행에는 사용하지 않은 ID를 지정합니다.
EXECUTE = False
KEEP_RAW_RESULTS = False  # True면 두 arm의 개발 임베딩 및 최종 backbone 보존
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in (".", ".."):
    raise ValueError("RUN_ID에는 단일 폴더 이름을 사용하세요.")
diagnostics = MODEL_ROOT / "facex/diagnostics"
parent = diagnostics / "parent_weight_decay_zero_v1/checkpoint/pretrain_arcface_seed8972/pretrain_mobilefacenet_1024d.pt"
output = diagnostics / "survface_input_pilot" / RUN_ID
print(ARMS, SIZES, GATE, output, sep="\n")

### 두 조건 실행

입력은 SurvFace 원본 training crop입니다. 5,319명으로 이미 학습한 s7e 가중치를 사용하면 개발 인물과 겹치므로, 두 조건을 VGGFace2 Parent에서 동일하게 시작합니다.

In [ ]:
if EXECUTE:
    run_pilot(DATASET_ROOT, parent, output, keep_raw_results=KEEP_RAW_RESULTS)

### 결과 읽기

TPIR과 실제 FPIR을 함께 확인합니다. 모델 선택 기준은 첫 셀에 표시됩니다. [분모, 신뢰구간, 원본 보존 및 재실행 안내](HOW_TO_READ.md)

In [ ]:
if (output / "COMPLETED").is_file():
    result = read_summary(output)
    for arm, metrics in result["arms"].items():
        print(arm, {k: metrics[k] for k in ("tp", "registered_n", "tpir", "fp", "unknown_n", "realized_fpir")})
    print("pad − stretch (%p)", result["delta_tpir_pp"], result["paired_registered_id_ci95_pp"])
    print(result["next_step"], output / result["guide"])
else:
    print("지정한 ID에 완료된 대조 결과가 없습니다.")